<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "242"
iso_country = "Fiji"
year = 2025
import_portfolio = "h6_ag6_by_sea_M*.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_DX*.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 6.200*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Fiji': 

Number of trade records present in dataset: 25658
Number of partner countries present in dataset: 150
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$2630.1m
Total weight within the dataset: 2171.1 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,242,Fiji,528,Netherlands,Import,Sea,901380,"901380 | Optical devices, app",9013,"9013 | Lasers, other than l",90,"90 | Optical, photographi",48315,"Liquid crystal devices n.e.c.; lasers, except ...",Other Manufacturing,41,OME - Machinery and equipment n.e.c.,2600.342625,NaN,2600.342625,u,5.75,NaN,32.504283,18430.0,11451.86753,372.234741,['Container'],['Container'],1.696938,1.693692,1.694024,1.694024,1.695472,1.695472,1.777071,3.174268,1.942213,1.822568,1.827838,1.869940,1.739166,3.098387,2.055160,1.889326,1.843989,1.930298,1.705921,3.401786,2.556729,2.238453,2.067821,2.013214,1.673293,3.606933,2.965002,2.757317,2.211551,2.159340,1.641280,3.678554,3.538020,3.582697,2.379284,2.269016,1.608625,3.601201,3.415519,3.724880,2.822344,2.311312,1.577862,3.186662,3.342383,3.494838,2.564950,2.338678,1.546494,2.929716,2.969918,3.578251,2.983602,2.354942,48315
1,1,242,Fiji,528,Netherlands,Import,Sea,901910,901910 | Mechano-therapy appl,9019,"9019 | Mechano-therapy, mas",90,"90 | Optical, photographi",4816,Mechano-therapy appliances; massage apparatus;...,Other Manufacturing,41,OME - Machinery and equipment n.e.c.,15.984780,NaN,15.984780,kg,1.47,1.47,0.561066,18430.0,11451.86753,6.425251,['Container'],['Container'],0.029291,0.029235,0.029241,0.029241,0.029266,0.029266,0.030675,0.054792,0.033525,0.031460,0.031551,0.032278,0.030020,0.053482,0.035475,0.032612,0.031830,0.033319,0.029446,0.058719,0.044132,0.038639,0.035693,0.034751,0.028883,0.062260,0.051180,0.047595,0.038174,0.037273,0.028331,0.063497,0.061071,0.061842,0.041070,0.039166,0.027767,0.062161,0.058956,0.064296,0.048717,0.039896,0.027236,0.055006,0.057694,0.060325,0.044274,0.040369,0.026694,0.050571,0.051265,0.061765,0.051501,0.040649,4816


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $519.2m (19.7%), ... 

                             cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                            
10 | Cereals               67661620.5  195037411.6      2.6      9.0
04 | Dairy produce; birds  63451797.6   28484980.6      2.4      1.3
21 | Miscellaneous edible  55648513.1   14290224.6      2.1      0.7
15 | Animal, vegetable or  39465135.3   32902196.7      1.5      1.5
07 | Vegetables and certa  38889264.2   74207748.7      1.5      3.4
22 | Beverages, spirits a  38647036.5   39477137.4      1.5      1.8
02 | Meat and edible meat  27607222.2    6807468.2      1.0      0.3
23 | Food industries, res  27008883.4   35924849.7      1.0      1.7
20 | Preparations of vege  24360572.9   16552106.2      0.9      0.8
19 | Preparations of cere  21945590.1    8003304.6      0.8      0.4
16 | Meat, fish, crustace  20653598.7    4387476.3      0.8      0.2
12

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [7]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $454.3m (17.3%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Agriculture       237789440.7  215490923.6      9.0      9.9
Food & Beverages  213496042.6  145316564.0      8.1      6.7
Fishing             3000894.6    1148624.7      0.1      0.1



### 2.1.3 GTAP (16x headings)

In [5]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $415.4m (15.8%), ... 

                                                 cifvalue  vol_kg_vtw  \
gtap_desc                                                               
OFD - Food products n.e.c.                    166590281.1  73509568.3   
MIL - Dairy products                           60933991.6  24560615.5   
V_F - Vegetables, fruit, nuts                  48260565.1  82102457.6   
VOL - Vegetable oils and fats                  47244332.7  53051636.3   
PCR - Processed rice                           26723552.3  46418810.4   
CMT - Bovine meat prods                        22567620.3   4328817.1   
OCR - Crops n.e.c.                             19919126.4   6748502.1   
SGR - Sugar                                    11113873.0  24302957.0   
OMT - Meat products n.e.c.                     10866042.6   5278441.6   
OSD - Oil seeds                                  682841.6    922215.6   
GRO - Cereal grains n.e.c.                       245323.3 

## 2.2 Comparison of results

In [8]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 19.7% 
GTAP 15.8% 
EORA 17.3%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [9]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [10]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_27816/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,242,Fiji,528,Netherlands,Import,Sea,901380,"901380 | Optical devices, app",9013,"9013 | Lasers, other than l",90,"90 | Optical, photographi",48315,"Liquid crystal devices n.e.c.; lasers, except ...",Other Manufacturing,41,OME - Machinery and equipment n.e.c.,2600.342625,NaN,2600.342625,u,5.75,NaN,32.504283,18430.0,11451.86753,372.234741,['Container'],['Container'],1.696938,1.693692,1.694024,1.694024,1.695472,1.695472,1.777071,3.174268,1.942213,1.822568,1.827838,1.869940,1.739166,3.098387,2.055160,1.889326,1.843989,1.930298,1.705921,3.401786,2.556729,2.238453,2.067821,2.013214,1.673293,3.606933,2.965002,2.757317,2.211551,2.159340,1.641280,3.678554,3.538020,3.582697,2.379284,2.269016,1.608625,3.601201,3.415519,3.724880,2.822344,2.311312,1.577862,3.186662,3.342383,3.494838,2.564950,2.338678,1.546494,2.929716,2.969918,3.578251,2.983602,2.354942,48315,NaN
1,1,242,Fiji,528,Netherlands,Import,Sea,901910,901910 | Mechano-therapy appl,9019,"9019 | Mechano-therapy, mas",90,"90 | Optical, photographi",4816,Mechano-therapy appliances; massage apparatus;...,Other Manufacturing,41,OME - Machinery and equipment n.e.c.,15.984780,NaN,15.984780,kg,1.47,1.47,0.561066,18430.0,11451.86753,6.425251,['Container'],['Container'],0.029291,0.029235,0.029241,0.029241,0.029266,0.029266,0.030675,0.054792,0.033525,0.031460,0.031551,0.032278,0.030020,0.053482,0.035475,0.032612,0.031830,0.033319,0.029446,0.058719,0.044132,0.038639,0.035693,0.034751,0.028883,0.062260,0.051180,0.047595,0.038174,0.037273,0.028331,0.063497,0.061071,0.061842,0.041070,0.039166,0.027767,0.062161,0.058956,0.064296,0.048717,0.039896,0.027236,0.055006,0.057694,0.060325,0.044274,0.040369,0.026694,0.050571,0.051265,0.061765,0.051501,0.040649,4816,NaN


## 3.3 Inspect the output

In [11]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $332.9m of $2630.1m (12.7%)
Volume: 398.1 of 2171.1 million kilograms (18.3%)
Value-to-Weight: 0.84 from 1.21 US$ per kg (+-31.0%)
Average distance to market (volume-weighted): 2960.1 from 4543.5 nm (+-34.9%)

            food_group  N_com    cifvalue   vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    549  99262365.5   72097603.1      3.8      3.3
1                Dairy     65  62928924.1   28263721.1      2.4      1.3
2           Vegetables    337  42631861.2   77004624.3      1.6      3.5
3  Unprocessed Cereals     37  40937698.0  148617958.4      1.6      6.8
4        Fats and Oils    134  33929480.4   26440465.0      1.3      1.2
5        Meat and Prep     44  23359698.6    6931624.4      0.9      0.3
6      Sugar and Honey    109  16723215.6   27672817.9      0.6      1.3
7               Fruits    188  13098020.9   11089777.4      0.5      0.5



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [12]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,25720074.4,45867185.2,28099000.5,26368448.4,26444507.2,27057795.9,23762200.5,53128048.8,51116227.4,51779632.3,34406523.1,32810150.3,22395566.8,42346185.0,42922326.2,51681140.9,43104663.2,34048588.8
1,Fats and Oils,53037854.5,89246240.4,57165017.9,53648255.0,53792567.9,55341841.2,49532662.2,101415191.9,98995856.9,101536541.8,68831089.0,65530015.1,47086081.6,83268602.8,84023370.5,98824199.4,83395889.2,67706485.0
2,Fruits,13086341.9,23310212.3,14290507.8,13413211.2,13451535.3,13764697.6,12092846.8,27002610.6,25968736.1,26314384.9,17497101.5,16682764.6,11399449.8,21535065.3,21828302.2,26264218.4,21896159.0,17311937.8
3,Meat and Prep,5083757.7,9080791.6,5556188.8,5213913.7,5228988.4,5349431.4,4695292.9,10523426.0,10121395.7,10249205.3,6806539.9,6491088.6,4424134.5,8381187.9,8496196.8,10236485.9,8535343.8,6736903.7
4,Processed Cereals,174602936.3,309074694.1,190310539.7,178722706.8,179222778.0,183485806.1,161540590.6,357781221.6,344090817.1,349200567.0,232758225.0,221784831.3,152426688.3,285915997.8,289772098.4,347845516.6,290012375.2,230060865.2
5,Sugar and Honey,48053057.1,82234351.2,51870855.7,48864804.9,49000158.0,50275526.9,44731036.0,96228794.9,92321029.5,94126447.9,63742412.0,59940413.6,42412710.2,77306341.3,78266880.7,92815298.7,77780444.4,61993666.9
6,Unprocessed Cereals,152905127.2,272988790.8,167090117.5,156804157.6,157257328.6,160885006.7,141234388.7,316396113.5,304299763.6,308165506.2,204701246.6,195180360.8,133087930.7,252011983.6,255466290.6,307738578.3,256612919.0,202563414.6
7,Vegetables,76861436.8,136419738.8,83839497.8,78716088.1,78940201.5,80801662.0,71076569.9,158088256.5,152003640.2,154142079.8,102630809.3,97784373.9,67034451.7,126142338.6,127870579.7,153648597.4,128112498.3,101443935.7


### 3.3.2 Total FEM costs by scenario (% cif)

In [13]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.32,0.04,0.01,0.01,0.02,0.47,0.43,0.45,0.17,0.14,0.32,0.33,0.47,0.33,0.19
1,Fats and Oils,1.07,0.12,0.02,0.02,0.07,1.53,1.46,1.53,0.57,0.47,1.07,1.09,1.52,1.07,0.61
2,Fruits,0.78,0.09,0.02,0.03,0.05,1.14,1.06,1.09,0.41,0.35,0.77,0.80,1.13,0.80,0.45
3,Meat and Prep,0.17,0.02,0.01,0.01,0.01,0.25,0.23,0.24,0.09,0.08,0.17,0.17,0.25,0.18,0.10
4,Processed Cereals,1.35,0.16,0.04,0.05,0.09,1.98,1.84,1.89,0.72,0.61,1.34,1.38,1.97,1.39,0.78
5,Sugar and Honey,2.04,0.23,0.05,0.06,0.13,3.08,2.85,2.95,1.14,0.91,2.09,2.14,3.01,2.11,1.17
6,Unprocessed Cereals,2.93,0.35,0.10,0.11,0.19,4.28,3.98,4.08,1.55,1.32,2.91,2.99,4.27,3.02,1.70
7,Vegetables,1.40,0.16,0.04,0.05,0.09,2.04,1.90,1.95,0.74,0.63,1.39,1.43,2.03,1.43,0.81
